<a href="https://colab.research.google.com/github/mango208014-ai/Green-GML-Profiler/blob/main/metrics_classical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import psutil
import os
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Define the HardwareProfiler class (from Week 2)
class HardwareProfiler:
    def __init__(self):
        self.process = psutil.Process(os.getpid())
        self.start_time = None
        self.start_ram = None
        self.peak_ram = 0

    def start(self):
        self.peak_ram = self.process.memory_info().rss
        self.start_time = time.time()

    def stop(self):
        end_time = time.time()
        current_ram = self.process.memory_info().rss
        if current_ram > self.peak_ram:
            self.peak_ram = current_ram

        latency = end_time - self.start_time
        peak_ram_mb = self.peak_ram / (1024 * 1024)
        power_kw = 0.015
        carbon_gco2e = (latency / 3600) * power_kw * 400

        return {
            "latency_seconds": round(latency, 4),
            "peak_ram_mb": round(peak_ram_mb, 2),
            "carbon_gco2e": round(carbon_gco2e, 6)
        }

profiler = HardwareProfiler()

# 2. Recreate the Dataset Pipeline & Split (from Week 1)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# 3. Execute 20 profiling trials for Logistic Regression and SVM
models = {
    "LogisticRegression": LogisticRegression(max_iter=200),
    "SVM": SVC()
}

results = []
num_trials = 20

for name, model in models.items():
    print(f"Running 20 trials for {name}...")
    for trial in range(num_trials):
        profiler.start()

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        metrics = profiler.stop()

        results.append({
            "model": name,
            "trial": trial + 1,
            "accuracy": round(acc, 4),
            "latency_seconds": metrics["latency_seconds"],
            "peak_ram_mb": metrics["peak_ram_mb"],
            "carbon_gco2e": metrics["carbon_gco2e"]
        })

# 4. Save outcomes into metrics_classical.csv
df_metrics = pd.DataFrame(results)
df_metrics.to_csv("metrics_classical.csv", index=False)

print("\nWeek 3 profiling complete! First few rows of metrics_classical.csv:")
print(df_metrics.head())

Running 20 trials for LogisticRegression...
Running 20 trials for SVM...

Week 3 profiling complete! First few rows of metrics_classical.csv:
                model  trial  accuracy  latency_seconds  peak_ram_mb  \
0  LogisticRegression      1    0.9722           0.1571       247.76   
1  LogisticRegression      2    0.9722           0.0821       247.76   
2  LogisticRegression      3    0.9722           0.0419       247.77   
3  LogisticRegression      4    0.9722           0.0751       247.77   
4  LogisticRegression      5    0.9722           0.1109       247.77   

   carbon_gco2e  
0      0.000262  
1      0.000137  
2      0.000070  
3      0.000125  
4      0.000185  
